# 02 — Baseline Model

Train a Random Forest IDS model, evaluate it on clean test data,
and inspect feature importances.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 1. Load and preprocess data

In [ ]:
from src.data_pipeline import load_data, auto_detect_dataset

data_path, _ = auto_detect_dataset()
X_train, X_test, y_train, y_test, scaler = load_data(data_path, n_features=20)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Attack fraction (test): {y_test.mean()*100:.1f}%')

## 2. Train Random Forest

In [ ]:
from src.model import train_random_forest, train_logistic_regression, evaluate

rf = train_random_forest(X_train, y_train, n_estimators=100, max_depth=10)
lr = train_logistic_regression(X_train, y_train)
print('Models trained.')

## 3. Evaluate

In [ ]:
rf_metrics = evaluate(rf, X_test, y_test)
lr_metrics = evaluate(lr, X_test, y_test)

print('=== Random Forest ===')
for k, v in rf_metrics.items():
    if k != 'confusion_matrix':
        print(f'  {k:20s}: {v*100:.1f}%' if isinstance(v, float) else f'  {k}: {v}')

print('\n=== Logistic Regression ===')
for k, v in lr_metrics.items():
    if k != 'confusion_matrix':
        print(f'  {k:20s}: {v*100:.1f}%' if isinstance(v, float) else f'  {k}: {v}')

## 4. Confusion matrix — Random Forest

In [ ]:
import numpy as np
from src.visualizer import plot_confusion_matrix

cm = np.array(rf_metrics['confusion_matrix'])
fig = plot_confusion_matrix(cm, 'Random Forest Baseline (No Attack)',
                             'confusion_matrix_baseline.png', save=True)
plt.show()

## 5. Feature importances

In [ ]:
importances = rf.feature_importances_
feat_names = [f'Feature {i}' for i in range(len(importances))]
idx = np.argsort(importances)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(idx)), importances[idx], color='steelblue', edgecolor='black')
ax.set_xticks(range(len(idx)))
ax.set_xticklabels([feat_names[i] for i in idx], rotation=45, ha='right')
ax.set_title('Top 15 Feature Importances — Random Forest')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'plots', 'feature_importances.png'), dpi=120, bbox_inches='tight')
plt.show()